<a href="https://colab.research.google.com/github/SeloJumadiko/-data-science-2026/blob/main/Pertemuan12_%5BSelo_Jumadiko%5D_%5B240401010276%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
# Langkah 1: Generate dan Eksplorasi Dataset Transaksi
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [24]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [25]:
# Langkah 2: One-Hot Encoding Transaksi

In [26]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [27]:
# Langkah 3: Cari Frequent Itemset dengan Apriori

In [28]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Menggunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [29]:
# Langkah 4: Bentuk & Saring Aturan Asosiasi

In [30]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))

         antecedents consequents  support  confidence      lift
8        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
13  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
9      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115




*   Aturan yang paling kuat adalah (Keju, Teh) -> (Telur) dengan nilai Lift: 2.38. Lift di atas 1 menunjukkan hubungan positif yang kuat,ini artinya pelanggan yang membeli Keju dan Teh memiliki kemungkinan 2,38 kali lebih besar untuk membeli Telur dibandingkan pelanggan rata-rata.
*   Aturan (Roti) -> (Selai) memiliki Lift: 1.32 dan Confidence: 0.68 (68%). Ini sangat masuk akal secara bisnis karena produk tersebut bersifat komplementer. Selain itu, aturan (Roti, Gula) -> (Selai) memiliki Confidence: 1.0 (100%), yang berarti setiap kali Roti dan Gula dibeli bersama, Selai selalu ada di keranjang tersebut.



In [31]:
# Langkah 5: Rekomender Sederhana dengan Content-Based Filtering

In [32]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
'produk': produk,
'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [ ]:
# Langkah 6: Bandingkan Kedua Pendekatan

In [34]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Dalam kasus ini hasilnya konsisten karena :

  1. Association Rules merekomendasikan Selai untuk pembelian Roti (dengan Lift > 1.3), yang berarti ada hubungan statistik yang kuat di data transaksi.
  2. Content-Based Filtering juga menempatkan Selai di urutan teratas karena keduanya berada dalam kategori yang sama ('Bakery').
  3. Perbedaannya: Content-Based juga menyarankan Sereal dan Susu karena kemiripan atribut (kategori), meskipun mungkin di data transaksi nyata, orang yang membeli Roti belum tentu membeli Sereal.

Menggunakan Association Rules Jika:
  1. Tujuan Anda adalah Cross-selling (menjual barang yang berbeda tapi sering dibeli bersama, misal: Pasta dan Saus Tomat).
  2. Jika memiliki data transaksi yang besar dan ingin menangkap kebiasaan riil pelanggan.

Menggunakan Content-Based Jika:
  1. Tujuannya adalah mencari Alternatif produk (misal: pelanggan mencari roti merk A yang habis, Anda tawarkan roti merk B).
  2. Sedang memiliki produk baru di katalog yang belum punya riwayat transaksi.
  
Menggunakan Hybrid (Menggabungkan Keduanya) Jika:
  1. Menginginkan sistem yang lebih canggih (seperti Netflix atau Amazon).
